In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

In [6]:
df = pd.read_csv("Sales.csv")

In [8]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [9]:
df.head()
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [43]:
#Replace Dirty Text with NaN
#This will convert all fake values into real missing values.
df.replace(["ERROR", "UNKNOWN", ""], np.nan, inplace=True)

In [44]:
item_price_map = {
    "Coffee": 2.0,
    "Cake": 3.0,
    "Cookie": 1.0,
    "Salad": 5.0,
    "Smoothie": 4.0,
    "Sandwich": 4.0,
    "Juice": 3.0
}
#created a price list for finding out unknown for price 

In [45]:
df.loc[:, "Price Per Unit"] = df["Item"].map(item_price_map).fillna(df["Price Per Unit"])
# what this code does is replace price unknown by matching the fix price of itemns

In [46]:
# Step 1: Reverse mapping
price_to_items = {}
for item, price in item_price_map.items():
    price_to_items.setdefault(price, []).append(item)

In [47]:
# Step 2: Keep only unique prices
unique_price_map = {
    price: items[0]
    for price, items in price_to_items.items()
    if len(items) == 1
}

In [48]:
# Step 3: Replace Unknown safely
df.loc[df["Item"] == "Unknown", "Item"] = (
    df.loc[df["Item"] == "Unknown", "Price Per Unit"]
      .map(unique_price_map)
)

In [49]:
df["Item"].value_counts().get("Unknown Item", 0)

np.int64(876)

In [50]:
cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

#Convert ALL numeric columns first

In [51]:
mask = (
    df["Quantity"].isna() &
    df["Price Per Unit"].notna() &
    (df["Price Per Unit"] != 0) &
    df["Total Spent"].notna()
)

df.loc[mask, "Quantity"] = (
    df.loc[mask, "Total Spent"] / df.loc[mask, "Price Per Unit"]
)  
#Avoid division by ZERO
#Some prices may be 0 or NaN

In [52]:
df["Quantity"] = df["Quantity"].fillna(1)
#Final fallback for remaining Quantity

In [53]:
# STEP 7: FIX TOTAL SPENT
# ==============================
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

In [54]:
price_to_item = {
    2.0: "Coffee",
    3.0: "Cake",
    1.0: "Cookie",
    5.0: "Salad",
    4.0: "Sandwich"   # default for price 4
}


In [55]:
mask = df["Item"].str.lower() == "unknown item"

df.loc[mask, "Item"] = df.loc[mask, "Price Per Unit"].map(price_to_item)


In [56]:
# STEP 9: CLEAN CATEGORICAL COLUMNS
# ==============================
df.loc[df["Payment Method"].str.lower() == "unknown", "Payment Method"] = "UPI Payment"
df.loc[df["Location"].str.lower() == "unknown", "Location"] = "Online / Unspecified"

df = df.fillna({
    "Payment Method": "UPI Payment",
    "Location": "Online / Unspecified",
    "Item": "Unknown Item"
})


In [57]:
# STEP 10: HANDLE DATE COLUMN
# ==============================
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
df = df[df["Transaction Date"].notna()].copy()


In [58]:
#STEP 11: CREATE DATE FEATURES
# ==============================
df["Year"] = df["Transaction Date"].dt.year
df["Month"] = df["Transaction Date"].dt.month
df["Month Name"] = df["Transaction Date"].dt.month_name()
df["Year_Month"] = df["Transaction Date"].dt.to_period("M")

In [59]:
# STEP 12: DROP INVALID ROWS
# ==============================
df = df.dropna(subset=["Quantity", "Price Per Unit"])

In [60]:
# STEP 13: FINAL DATA CONSISTENCY CHECK
# ==============================
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

In [61]:
df.duplicated().sum()

np.int64(0)

In [62]:
(df["Quantity"] <= 0).sum()
(df["Price Per Unit"] <= 0).sum()
(df["Total Spent"] < 0).sum()


np.int64(0)

In [63]:
(df["Total Spent"] != df["Quantity"] * df["Price Per Unit"]).sum()


np.int64(0)

In [64]:
df["Payment Method"].value_counts()
df["Location"].value_counts()
df["Item"].value_counts()


Item
Cake            1305
Sandwich        1277
Coffee          1239
Salad           1214
Cookie          1147
Juice           1124
Smoothie        1048
Tea              965
Unknown Item     108
Name: count, dtype: int64

In [65]:
df["Transaction Date"].min()
df["Transaction Date"].max()


Timestamp('2023-12-31 00:00:00')

In [66]:
df.to_csv("Sales_Cleaned.csv", index=False)